In [ ]:
!pip install stable-baselines3 gymnasium pandas matplotlib

In [ ]:
import os
os.environ["TRITON_DISABLE_LINE_INFO"] = "1"

import torch
import triton
import triton.language as tl
import time
import numpy as np
import gymnasium as gym
import pandas as pd
from stable_baselines3 import PPO


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
print("Torch:", torch.__version__)
print("Triton:", triton.__version__)
print("CUDA available:", torch.cuda.is_available())

Torch: 2.10.0+cu128
Triton: 3.6.0
CUDA available: True


In [ ]:
import triton
import triton.language as tl

@triton.jit
def matmul_kernel(
    a_ptr, b_ptr, c_ptr,
    M, N, K,
    stride_am, stride_ak,
    stride_bk, stride_bn,
    stride_cm, stride_cn,
    BLOCK_M: tl.constexpr,
    BLOCK_N: tl.constexpr,
    BLOCK_K: tl.constexpr,
):

    pid = tl.program_id(0)

    num_pid_m = tl.cdiv(M, BLOCK_M)
    pid_m = pid // num_pid_m
    pid_n = pid % num_pid_m

    offs_m = pid_m * BLOCK_M + tl.arange(0, BLOCK_M)
    offs_n = pid_n * BLOCK_N + tl.arange(0, BLOCK_N)
    offs_k = tl.arange(0, BLOCK_K)

    acc = tl.zeros((BLOCK_M, BLOCK_N), dtype=tl.float32)

    for k in range(0, K, BLOCK_K):
        k_ids = k + offs_k

        a = tl.load(
            a_ptr + offs_m[:, None] * stride_am + k_ids[None, :] * stride_ak,
            mask=(offs_m[:, None] < M) & (k_ids[None, :] < K),
            other=0.0
        )

        b = tl.load(
            b_ptr + k_ids[:, None] * stride_bk + offs_n[None, :] * stride_bn,
            mask=(k_ids[:, None] < K) & (offs_n[None, :] < N),
            other=0.0
        )

        acc += tl.dot(a, b)

    tl.store(
        c_ptr + offs_m[:, None] * stride_cm + offs_n[None, :] * stride_cn,
        acc,
        mask=(offs_m[:, None] < M) & (offs_n[None, :] < N)
    )


In [ ]:
import time
import torch

def run_timed(BM, BN, BK, M, N, K):

    A = torch.rand((M, K), device="cuda")
    B = torch.rand((K, N), device="cuda")
    C = torch.empty((M, N), device="cuda")

    grid = lambda META: (
        triton.cdiv(M, META['BLOCK_M']) *
        triton.cdiv(N, META['BLOCK_N']),
    )

    torch.cuda.synchronize()
    start = time.time()

    matmul_kernel[grid](
        A, B, C,
        int(M), int(N), int(K),
        A.stride(0), A.stride(1),
        B.stride(0), B.stride(1),
        C.stride(0), C.stride(1),
        BLOCK_M=int(BM),
        BLOCK_N=int(BN),
        BLOCK_K=int(BK),
        num_warps=4,
        num_stages=2
    )

    torch.cuda.synchronize()
    return time.time() - start

In [ ]:
import torch

print(run_timed(32, 32, 16, 512, 512, 512))
print(run_timed(64, 64, 32, 512, 512, 512))

3.336540460586548
0.4627063274383545


In [ ]:
import pandas as pd

workloads = pd.DataFrame({
    "M": [256, 512, 1024, 512],
    "N": [256, 512, 1024, 1024],
    "K": [256, 512, 1024, 512]
})

In [ ]:
import gymnasium as gym
import numpy as np

class TritonCRLEnv(gym.Env):

    def __init__(self, workloads):
        super().__init__()
        self.workloads = workloads

        self.action_space = gym.spaces.Box(
            low=np.array([16,16,16, 16,16,16]),
            high=np.array([128,128,64, 128,128,64]),
            dtype=np.float32
        )

        self.observation_space = gym.spaces.Box(
            low=0, high=5000, shape=(3,), dtype=np.float32
        )

    def reset(self, seed=None):
        row = self.workloads.sample(1).iloc[0]
        self.M = int(row["M"])
        self.N = int(row["N"])
        self.K = int(row["K"])
        return np.array([self.M, self.N, self.K]), {}

    def step(self, action):

        BM1, BN1, BK1 = int(action[0]), int(action[1]), int(action[2])
        BM2, BN2, BK2 = int(action[3]), int(action[4]), int(action[5])

        t1 = run_timed(BM1, BN1, BK1, self.M, self.N, self.K)
        t2 = run_timed(BM2, BN2, BK2, self.M, self.N, self.K)

        reward = float(t2 - t1)   # difference-based reward

        done = True
        return np.array([self.M, self.N, self.K]), reward, done, False, {}

In [ ]:
!pip install stable-baselines3
from stable_baselines3 import PPO

env = TritonCRLEnv(workloads)

model = PPO(
    "MlpPolicy",
    env,
    verbose=1,
    device="cpu"
)

model.learn(total_timesteps=2000)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.0/188.0 kB 7.9 MB/s eta 0:00:00


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


----------------------------------
| rollout/           |           |
|    ep_len_mean     | 1         |
|    ep_rew_mean     | -1.84e-05 |
| time/              |           |
|    fps             | 392       |
|    iterations      | 1         |
|    time_elapsed    | 5         |
|    total_timesteps | 2048      |
----------------------------------


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:

def evaluate(model, env, episodes=10):
    rewards = []
    for _ in range(episodes):
        obs,_ = env.reset()
        action,_ = model.predict(obs)
        _, reward, _, _, _ = env.step(action)
        rewards.append(reward)
    return np.mean(rewards)

print("Average comparative reward:", evaluate(model, env))

Average comparative reward: -2.715587615966797e-05
